# 🧠 Physics-Informed Neural Networks (PINNs) — Tutorial 1
## Solving a 1D First-Order ODE from Scratch

---

### 🎯 What you will learn in this tutorial

1. How to turn an ODE into a **PINN loss function**
2. How to compute **one derivative** with `torch.autograd.grad`
3. How to build a **neural network** from scratch in PyTorch
4. What **collocation points** are and why `requires_grad=True` matters
5. How to **train** and **validate** against an exact solution

---

### 📐 The Problem We Are Solving

We solve the simplest possible ODE a PINN can tackle — a **first-order linear IVP**:

$$
\frac{du}{dx} = f(x), \quad x \in [0, 1]
$$

with initial condition:

$$
u(0) = 0
$$

and source term:

$$
f(x) = \cos(\pi x)
$$

The **exact analytical solution** is simply the integral of $f$:

$$
u(x) = \frac{\sin(\pi x)}{\pi}
$$

> **Why this problem?**  
> It needs exactly **one** `autograd.grad` call — no second derivative, no nonlinearity. The entire focus is on understanding the PINN setup: residual, loss, training loop. Everything in Tutorial 2 and beyond builds on exactly this structure.

---

### 🗺️ Series progression

| Tutorial | ODE / PDE | Derivative order | New concept |
|---|---|---|---|
| **1 (this)** | `du/dx = cos(πx)` | 1st order | PINN from scratch |
| 2 | `-u'' = sin(πx)` | 2nd order | Two `autograd.grad` calls |
| 3 | Burgers (nonlinear) | 2nd order + nonlinear | L-BFGS, viscosity |
| 4 | 2D Poisson | 2D partial derivatives | meshgrid, heatmaps |

---

## 🧩 Step 1 — Imports

Only three packages. No special PINN library — everything from scratch.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

---

## 🧩 Step 2 — Define the Exact Solution and Source Term

We write two functions:
- `u_exact` — the analytical answer, used **only** for validation at the end
- `f_source` — the right-hand side of the ODE, used **during training** to form the residual

> In a real problem you will not have `u_exact`. You only have the ODE and the boundary/initial condition. We include it here purely to measure how well the PINN did.

In [ ]:
def u_exact(x):
    """
    Exact solution:  du/dx = cos(pi*x),  u(0) = 0
    Integrating:     u(x)  = sin(pi*x) / pi
    """
    return np.sin(np.pi * x) / np.pi


def f_source(x):
    """
    Source term (right-hand side of the ODE): f(x) = cos(pi*x)
    x must be a PyTorch tensor — this is called inside the training loop.
    """
    return torch.cos(torch.pi * x)


# Sanity-check plot
x_plot = np.linspace(0, 1, 300)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(x_plot, u_exact(x_plot), 'k-', lw=2)
axes[0].set_title('Exact solution  u(x) = sin(πx) / π')
axes[0].set_xlabel('x'); axes[0].set_ylabel('u(x)'); axes[0].grid(alpha=0.3)

axes[1].plot(x_plot, np.cos(np.pi * x_plot), 'steelblue', lw=2)
axes[1].set_title('Source term  f(x) = cos(πx)  [= du/dx]')
axes[1].set_xlabel('x'); axes[1].set_ylabel('f(x)'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Note: f(x) is the derivative of u(x) — that is exactly the ODE.')

---

## 🧩 Step 3 — Build the Neural Network

The network takes one input ($x$) and outputs one value ($\hat{u}$):

$$
x \;\longrightarrow\; \underbrace{[\text{Linear} \to \tanh] \times 3}_{\text{hidden layers}} \;\longrightarrow\; \hat{u}(x)
$$

### Architecture table

| Parameter | Value | Reason |
|---|---|---|
| Input dim | 1 | scalar $x$ |
| Hidden layers | 3 | enough capacity for a smooth 1D function |
| Neurons | 32 | small — trains in seconds on CPU |
| Activation | `tanh` | smooth, differentiable — required for autograd |
| Output dim | 1 | scalar $\hat{u}(x)$ |

> **Why tanh and not ReLU?**  
> We need to differentiate the network output with respect to $x$. `tanh` is smooth everywhere so its derivative is well-defined. `ReLU` is piecewise linear — its derivative is a step function and barely informative.

In [ ]:
class PINN(nn.Module):
    """
    Simple fully-connected network for 1D PINN problems.
    Input:  x  (shape: N x 1)
    Output: u  (shape: N x 1)
    """

    def __init__(self, n_hidden=3, n_neurons=32):
        super().__init__()

        layers = []

        # First layer: 1 input → n_neurons
        layers.append(nn.Linear(1, n_neurons))
        layers.append(nn.Tanh())

        # Hidden layers: n_neurons → n_neurons
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(n_neurons, n_neurons))
            layers.append(nn.Tanh())

        # Output layer: n_neurons → 1  (no activation — raw output)
        layers.append(nn.Linear(n_neurons, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model = PINN(n_hidden=3, n_neurons=32).to(device)
print(model)
print(f'\nTotal trainable parameters: {sum(p.numel() for p in model.parameters())}')

---

## 🧩 Step 4 — Collocation Points

A PINN does not need labelled data `(x, u)`. It only needs **locations in the domain** where it can check how well the ODE is satisfied. These are called **collocation points**.

```
x=0 ●  ·  ·  ·  ·  ·  ·  ·  ·  · x=1
 IC      interior collocation points
```

The single most important line:  
```python
x_col.requires_grad_(True)
```

This tells PyTorch: *"track all operations involving this tensor so you can differentiate through them later."*  
Without it, `autograd.grad` raises an error — PyTorch has no record of how `u_pred` depended on `x`.

In [ ]:
N_COL = 100    # number of interior collocation points

# Interior: evenly spaced across [0, 1]
x_col = torch.linspace(0, 1, N_COL, dtype=torch.float32).view(-1, 1).to(device)
x_col.requires_grad_(True)   # ← non-negotiable

# Initial condition: u(0) = 0
x_ic  = torch.tensor([[0.0]], dtype=torch.float32).to(device)
u_ic  = torch.tensor([[0.0]], dtype=torch.float32).to(device)

print(f'Collocation points shape : {x_col.shape}')
print(f'requires_grad            : {x_col.requires_grad}')
print(f'IC point x=0, u=0        : ({x_ic.item()}, {u_ic.item()})')

# Quick layout plot
plt.figure(figsize=(10, 1.2))
plt.scatter(x_col.detach().numpy(), np.zeros(N_COL),
            s=20, c='steelblue', alpha=0.7, label=f'Collocation pts (N={N_COL})')
plt.scatter([0.0], [0.0], s=100, c='red', zorder=5, label='IC: u(0)=0')
plt.yticks([]); plt.xlabel('x')
plt.title('Collocation Point Layout')
plt.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

---

## 🧩 Step 5 — The ODE Residual (Core of the PINN)

The ODE says:

$$
\frac{du}{dx} - f(x) = 0
$$

If the network perfectly satisfies the ODE, the left-hand side is zero everywhere.  
We call this the **residual**:

$$
r(x) = \frac{d\hat{u}}{dx} - f(x)
$$

We compute $d\hat{u}/dx$ with `torch.autograd.grad` — **one call, one derivative**. This is the simplest possible autograd usage.

### What each argument does

```python
torch.autograd.grad(
    outputs  = u,                      # the tensor to differentiate
    inputs   = x,                      # differentiate w.r.t. this
    grad_outputs = torch.ones_like(u), # sum reduction (needed for vector outputs)
    create_graph = True                # keep graph so optimizer can backprop through it
)[0]                                   # [0] unpacks the tuple — one gradient per input
```

In [ ]:
def compute_ode_residual(model, x):
    """
    ODE residual:  r(x) = du/dx - f(x)

    If the network satisfies the ODE perfectly, r(x) = 0 everywhere.

    Args:
        model : PINN network
        x     : collocation points, shape (N, 1), requires_grad=True

    Returns:
        residual : Tensor, shape (N, 1)
    """
    u = model(x)          # forward pass: x → u_pred

    # Compute du/dx using automatic differentiation
    # This is EXACT — no finite difference approximation
    u_x = torch.autograd.grad(
        outputs      = u,
        inputs       = x,
        grad_outputs = torch.ones_like(u),
        create_graph = True    # must be True so the optimizer can backprop through u_x
    )[0]

    f = f_source(x)       # right-hand side: cos(pi*x)

    residual = u_x - f    # ODE: du/dx - f(x) = 0
    return residual


print('ODE residual function defined ✓')
print()
print('Breakdown:')
print('  u_x  = du/dx   computed by autograd  (exact, not approximate)')
print('  f    = cos(πx) evaluated at x        (the RHS we defined above)')
print('  r    = u_x - f                        (zero if ODE is satisfied)')

---

## 🧩 Step 6 — The Loss Function

The total PINN loss combines two terms:

$$
\mathcal{L} = \underbrace{\frac{1}{N}\sum_{i=1}^{N} r(x_i)^2}_{\mathcal{L}_{\text{ODE}}} + \lambda \cdot \underbrace{(\hat{u}(0) - 0)^2}_{\mathcal{L}_{\text{IC}}}
$$

| Term | What it enforces | When it is zero |
|---|---|---|
| $\mathcal{L}_{\text{ODE}}$ | The ODE must hold everywhere in $[0,1]$ | When `du/dx = f(x)` at all collocation points |
| $\mathcal{L}_{\text{IC}}$ | The initial condition $u(0) = 0$ | When the network outputs zero at $x=0$ |

$\lambda$ is the **weight** on the IC term. Setting it larger than 1 tells the optimizer: *"satisfying the IC is more important than minimising the ODE residual right now."*

> **Why does the IC need a separate term?**  
> The ODE `du/dx = f(x)` has infinitely many solutions — one for every constant of integration. The IC `u(0)=0` pins down the unique one we want. Without it the network could find `u(x) = sin(πx)/π + 5` and the ODE would still be satisfied.

In [ ]:
def compute_loss(model, x_col, x_ic, u_ic, lambda_ic=10.0):
    """
    Total PINN loss = L_ODE + lambda_ic * L_IC

    Args:
        model     : PINN network
        x_col     : interior collocation points (N, 1), requires_grad=True
        x_ic      : IC location x=0,  shape (1, 1)
        u_ic      : IC value    u=0,  shape (1, 1)
        lambda_ic : weight on IC loss

    Returns:
        loss_total, loss_ode (float), loss_ic (float)
    """
    # ODE residual loss
    residual = compute_ode_residual(model, x_col)
    loss_ode = torch.mean(residual ** 2)

    # IC loss: network must output 0 at x=0
    u_pred_ic = model(x_ic)
    loss_ic   = torch.mean((u_pred_ic - u_ic) ** 2)

    loss_total = loss_ode + lambda_ic * loss_ic
    return loss_total, loss_ode.item(), loss_ic.item()


# Quick check on untrained model
with torch.no_grad():
    u0 = model(x_col)

print(f'Untrained output range : [{u0.min():.3f}, {u0.max():.3f}]')
print(f'Target range           : [0.000,  {1/np.pi:.3f}]')
print()
print('Loss function defined ✓')

---

## 🧩 Step 7 — Training Loop

The training loop is identical for every PINN you will ever write:

```
for each epoch:
    1. zero_grad()          — clear gradients from last step
    2. compute_loss()       — forward pass + autograd residual
    3. loss.backward()      — backprop through everything
    4. optimizer.step()     — update weights
```

That's it. Same structure in Tutorials 2, 3, 4.

In [ ]:
N_EPOCHS    = 5000
LR          = 1e-3
LAMBDA_IC   = 10.0
LOG_EVERY   = 500

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {'epoch': [], 'loss_total': [], 'loss_ode': [], 'loss_ic': []}

print(f'{"Epoch":>8}  {"Loss Total":>14}  {"Loss ODE":>14}  {"Loss IC":>14}')
print('-' * 60)

for epoch in range(1, N_EPOCHS + 1):

    optimizer.zero_grad()

    loss_total, loss_ode, loss_ic = compute_loss(
        model, x_col, x_ic, u_ic, LAMBDA_IC
    )

    loss_total.backward()
    optimizer.step()

    if epoch % LOG_EVERY == 0 or epoch == 1:
        history['epoch'].append(epoch)
        history['loss_total'].append(loss_total.item())
        history['loss_ode'].append(loss_ode)
        history['loss_ic'].append(loss_ic)
        print(f'{epoch:>8}  {loss_total.item():>14.6e}  {loss_ode:>14.6e}  {loss_ic:>14.6e}')

print('\nTraining complete ✓')

---

## 🧩 Step 8 — Plot Loss History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(history['epoch'], history['loss_total'], 'b-o', ms=5, label='Total')
axes[0].semilogy(history['epoch'], history['loss_ode'],   'r--s', ms=5, label='ODE')
axes[0].semilogy(history['epoch'], history['loss_ic'],    'g:^',  ms=5, label='IC')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (log)')
axes[0].set_title('Training Loss History')
axes[0].legend(); axes[0].grid(True, which='both', alpha=0.3)

axes[1].semilogy(history['epoch'], history['loss_ode'], 'r-o', ms=5, label='ODE residual')
axes[1].semilogy(history['epoch'], history['loss_ic'],  'g-s', ms=5, label='IC error')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss (log)')
axes[1].set_title('ODE Loss vs IC Loss')
axes[1].legend(); axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('loss_history_t1.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 🧩 Step 9 — Compare PINN to Exact Solution

In [ ]:
x_test_np = np.linspace(0, 1, 500)
x_test    = torch.tensor(x_test_np, dtype=torch.float32).view(-1, 1).to(device)

model.eval()
with torch.no_grad():
    u_pred_np = model(x_test).cpu().numpy().flatten()

u_exact_np = u_exact(x_test_np)
error_np   = np.abs(u_pred_np - u_exact_np)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Solution
axes[0].plot(x_test_np, u_exact_np, 'k-',  lw=2.5, label='Exact: sin(πx)/π')
axes[0].plot(x_test_np, u_pred_np,  'r--', lw=2.0, label='PINN prediction')
axes[0].scatter(x_col.detach().cpu().numpy(), np.zeros(N_COL),
                c='steelblue', s=10, zorder=5, label='Collocation pts')
axes[0].scatter([0], [0], c='red', s=80, zorder=6, label='IC: u(0)=0')
axes[0].set_xlabel('x'); axes[0].set_ylabel('u(x)')
axes[0].set_title('PINN vs Exact Solution')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Error
axes[1].semilogy(x_test_np, error_np + 1e-12, 'purple', lw=2)
axes[1].fill_between(x_test_np, error_np + 1e-12, alpha=0.2, color='purple')
axes[1].set_xlabel('x'); axes[1].set_ylabel('|u_exact − u_PINN| (log)')
axes[1].set_title('Pointwise Absolute Error')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('pinn_vs_exact_t1.png', dpi=150, bbox_inches='tight')
plt.show()

l2_error  = np.sqrt(np.mean((u_pred_np - u_exact_np) ** 2))
max_error = np.max(error_np)
print(f'L2 error  : {l2_error:.4e}')
print(f'Max error : {max_error:.4e}')

---

## 🧩 Step 10 — Visualise the ODE Residual Field After Training

We evaluate $r(x) = d\hat{u}/dx - f(x)$ on a fine grid after training.  
A residual close to zero everywhere means the PINN has genuinely learned the ODE — not just memorised a curve.

In [ ]:
x_res = torch.linspace(0, 1, 300, dtype=torch.float32).view(-1, 1).to(device)
x_res.requires_grad_(True)

res_field = compute_ode_residual(model, x_res)
res_np    = res_field.detach().cpu().numpy().flatten()
x_res_np  = x_res.detach().cpu().numpy().flatten()

plt.figure(figsize=(8, 3.5))
plt.plot(x_res_np, res_np, 'darkorange', lw=2, label='r(x) = du/dx − f(x)')
plt.axhline(0, color='k', lw=0.8, ls='--')
plt.fill_between(x_res_np, res_np, alpha=0.15, color='darkorange')
plt.xlabel('x')
plt.ylabel('Residual')
plt.title('ODE Residual After Training\n(Should be close to zero everywhere)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ode_residual_t1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Max residual magnitude: {np.abs(res_np).max():.4e}')

---

## 🧩 Step 11 — What Happens Without the IC Term?

This is the most instructive experiment in Tutorial 1.

Set `lambda_ic = 0` and retrain. The ODE `du/dx = cos(πx)` will still be satisfied — but the network will pick an arbitrary constant of integration and the solution will be **vertically shifted**.

> This is why boundary/initial conditions are not optional in a PINN. The ODE alone does not uniquely determine the solution.

In [ ]:
# Train a fresh model with NO IC enforcement
torch.manual_seed(0)
model_no_ic = PINN().to(device)
opt_no_ic   = torch.optim.Adam(model_no_ic.parameters(), lr=1e-3)

for epoch in range(5000):
    opt_no_ic.zero_grad()
    loss_total, _, _ = compute_loss(
        model_no_ic, x_col, x_ic, u_ic,
        lambda_ic=0.0    # ← IC weight set to zero
    )
    loss_total.backward()
    opt_no_ic.step()

with torch.no_grad():
    u_no_ic = model_no_ic(x_test).cpu().numpy().flatten()

plt.figure(figsize=(8, 4))
plt.plot(x_test_np, u_exact_np, 'k-',  lw=2.5, label='Exact')
plt.plot(x_test_np, u_pred_np,  'b--', lw=2.0, label='PINN (with IC, λ=10)')
plt.plot(x_test_np, u_no_ic,    'r:',  lw=2.0, label='PINN (NO IC, λ=0)')
plt.xlabel('x'); plt.ylabel('u(x)')
plt.title('Effect of Removing the IC Loss Term\n(No IC → arbitrary vertical shift)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('no_ic_comparison_t1.png', dpi=150, bbox_inches='tight')
plt.show()

shift = u_no_ic[0]   # value at x=0 — should be 0 but isn't
print(f'Without IC:  u(0) = {shift:.4f}  (should be 0.0000)')
print(f'With    IC:  u(0) = {u_pred_np[0]:.4f}  (correctly pinned to 0)')

---

## 📚 Summary — What We Built

| Component | Detail |
|---|---|
| **ODE** | `du/dx = cos(πx)`,  IC: `u(0) = 0` |
| **Exact solution** | `sin(πx) / π` |
| **Network** | MLP: 1 → 32 → 32 → 32 → 1, `tanh` |
| **Residual** | `r(x) = u_x − f(x)` via one `autograd.grad` call |
| **Loss** | `L_ODE + 10 × L_IC` |
| **Optimizer** | Adam, lr=1e-3, 5000 epochs |
| **Key experiment** | Removing IC → arbitrary vertical shift |

---

## ➡️ What's Next — Tutorial 2 Preview

In **Tutorial 2** we step up to a **second-order ODE**:

$$
-\frac{d^2u}{dx^2} = \sin(\pi x), \quad u(0) = 0, \quad u(1) = 0
$$

New concepts:
- Two `autograd.grad` calls chained together
- Two boundary conditions instead of one IC
- Why `create_graph=True` on the **first** derivative is essential

Everything else — network, loss structure, training loop — stays identical.

---

## 🧩 Exercises

1. **Change the source term** — replace `cos(πx)` with `x²`. What is the new exact solution? Does the PINN still converge?
2. **Change the IC** — set `u(0) = 1`. Update the code and verify the PINN shifts the solution correctly.
3. **Reduce N_COL to 5** — how few collocation points can you use before the solution breaks?
4. **Remove `create_graph=True`** — what error do you get? Why?
5. **Try `lambda_ic = 0.01`** — how does the IC enforcement change compared to `lambda_ic = 10`?